# 04 · Preprocessing (Ludovico)

**Question:** which preprocessing should the models receive? Everything here **learns from the data** (medians, percentiles, category frequencies), so it lives inside a `Pipeline` and is fitted on the training fold only.

- **Part 1 · Missing values:** how to fill them, whether to flag first purchases, impute or drop.
- **Part 2 · Numeric columns:** skewness, scaling, outliers.
- **Part 3 · Categorical columns:** near-constant columns, rare levels, encoding.

**Method** (the one of the week 3 practical, *"Missing values: compare nine, then select"*):
- A **plan** says how each group of columns is filled. Every plan is scored on the **same splits**, with the same model.
- Plans are compared **in pairs, fold by fold**, against a reference: mean difference, its standard error (**paired SEM**) and the number of folds won.
- A difference smaller than about 2 SEM is read as **no difference**. Without a clear gain, the simplest plan is kept.

**Differences from the practical**
- Time-ordered splits with a 90-day gap instead of random splits (`validation_ludovico`).
- Here part of the missing values is **structural** (first purchases): it carries information, so it gets its own question.
- Metric: **ROC AUC**. It does not depend on the threshold, so it is less noisy than F1 for comparing plans.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>1. Where the missing values are</font>](#where)
* [<font color='#E8800A'>2. The splits every plan is scored on</font>](#splits)
* [<font color='#E8800A'>3. Fill plans: compare, then select</font>](#plans)
* [<font color='#E8800A'>4. First purchases: flag or not</font>](#structural)
* [<font color='#E8800A'>5. Impute or drop</font>](#drop)
* [<font color='#E8800A'>6. Effect on the official validation</font>](#effect)
* [<font color='#E8800A'>7. Summary of part 1</font>](#summary)
* [<font color='#E8800A'>8. Skewness and scaling</font>](#skew)
* [<font color='#E8800A'>9. Outliers</font>](#outliers)
* [<font color='#E8800A'>10. Categorical columns</font>](#categorical)
* [<font color='#E8800A'>11. The two recipes on the official validation</font>](#recipes)
* [<font color='#E8800A'>12. Summary</font>](#final)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

`%run` executes `validation_ludovico.ipynb` (which runs `cleaning_ludovico.ipynb`). This gives back the clean data, `X`/`y`, the split generator `time_splits`, the official folds (`schemes["time CV + gap"]`), `evaluate` and `best_threshold`.

In [1]:
%%capture
%run ./validation_ludovico.ipynb

In [2]:
import re
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (must come before IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

OFFICIAL = "time CV + gap"
official_folds = schemes[OFFICIAL]

# <font color='#E8800A'>1. Where the missing values are</font> <a class="anchor" id="where"></a>
[Back to TOC](#toc)

**Two kinds of missing values**, which need different answers:
- **Structural:** the value cannot exist. A first purchase (`prior_purchase_count = 0`) has no previous purchase, so no recency, no gap and no value statistics. A standard deviation needs at least 2 values, a gap standard deviation at least 3 purchases (2 gaps).
- **Scattered:** the value exists but is missing (≈3% of the categoricals, part of `spend_365d`, the values blanked in cleaning that could not be rebuilt).

The rule below says, for each history column, how many prior purchases it needs to exist.

In [3]:
# Minimum prior purchases for the value to exist: below it, a NaN is structural
NEEDS_PRIOR = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1, "prior_min_purchase_value": 1, "prior_max_purchase_value": 1,
    "prior_mean_gap_days": 2, "prior_std_purchase_value": 2,
    "prior_std_gap_days": 3,
}


def missing_kinds(df):
    """Two boolean frames (same shape as df): structural NaNs and scattered NaNs."""
    count = df["prior_purchase_count"]
    missing = df.isna()
    structural = pd.DataFrame(False, index=df.index, columns=df.columns)
    for col, needed in NEEDS_PRIOR.items():
        structural[col] = missing[col] & (count < needed)
    return structural, missing & ~structural


profile = {}
for name, frame in [("train", X), ("test", test.drop(columns=[c for c in NOT_FEATURES if c in test]))]:
    structural, scattered = missing_kinds(frame)
    profile[f"structural {name}"] = structural.sum()
    profile[f"scattered {name}"] = scattered.sum()
profile = pd.DataFrame(profile)
profile = profile[profile.sum(axis=1) > 0].sort_values("structural train", ascending=False)
profile

,structural train,scattered train,structural test,scattered test
prior_std_gap_days,1413,0,69,0
prior_std_purchase_value,1048,0,46,0
prior_mean_gap_days,1048,8,46,0
days_since_previous_purchase,603,8,22,0
prior_mean_purchase_value,603,0,22,0
prior_min_purchase_value,603,0,22,0
prior_max_purchase_value,603,42,22,6
spend_365d,0,156,0,36
payment_type,0,195,0,30
salesperson,0,194,0,30


In [4]:
structural_train, scattered_train = missing_kinds(X)
rows_scattered = scattered_train.any(axis=1)

print(f"first purchases (all structural NaNs): {(X['prior_purchase_count'] == 0).sum()} train rows")
print(f"rows with at least one scattered NaN:   {rows_scattered.sum()} train rows ({rows_scattered.mean():.1%})")

first purchases (all structural NaNs): 603 train rows
rows with at least one scattered NaN:   928 train rows (14.3%)


**Reading the table**
- **Most missing values are structural.** All the NaNs of `prior_mean/min_purchase_value` and almost all of the gap and std columns come from customers with too few prior purchases (603 first purchases, plus the rows with 1–2 prior purchases for the std columns).
- **The scattered ones** are concentrated in 4 categoricals (~195 rows each, ~3%), `spend_365d` (156) and `prior_max_purchase_value` (42, the inflated values blanked in cleaning). In total 928 train rows (14.3%) have at least one scattered NaN.
- Test has the same pattern in smaller numbers, so whatever is learned here applies to it too.

# <font color='#E8800A'>2. The splits every plan is scored on</font> <a class="anchor" id="splits"></a>
[Back to TOC](#toc)

**The official validation has only 3 folds.** Three paired differences are too few for a standard error. For these comparisons only, the same expanding time scheme with the same 90-day gap uses **half-year validation blocks from 2021**, giving 8 folds.

The official 3-fold scheme is kept for reporting results (section 6).

In [5]:
PREP_STARTS = pd.date_range("2021-01-01", "2024-07-01", freq="6MS")  # 8 half-year validation blocks
prep_folds = list(time_splits(dates, PREP_STARTS))

pd.DataFrame([describe_split(train.iloc[tr], train.iloc[va], f"fold {k}")
              for k, (tr, va) in enumerate(prep_folds, 1)]).set_index("scheme")[
    ["train rows", "val rows", "val period", "gap days", "seen customers", "val positive rate"]
].style.format({"seen customers": "{:.1%}", "val positive rate": "{:.3f}"})

,train rows,val rows,val period,gap days,seen customers,val positive rate
scheme,,,,,,
fold 1,3234,315,2021-01 -> 2021-06,93,91.1%,0.581
fold 2,3649,467,2021-07 -> 2021-12,95,86.9%,0.527
fold 3,4004,269,2022-01 -> 2022-06,91,94.4%,0.539
fold 4,4392,359,2022-07 -> 2022-12,95,87.5%,0.437
fold 5,4705,341,2023-01 -> 2023-06,91,92.7%,0.563
fold 6,5077,426,2023-07 -> 2023-12,92,92.0%,0.596
fold 7,5403,353,2024-01 -> 2024-06,91,93.5%,0.601
fold 8,5861,482,2024-07 -> 2025-02,92,94.2%,0.608


Every fold trains only on the past, keeps the 90-day gap and validates on 300–450 rows. Each fold is noisier than an official one, but there are 8 paired differences instead of 3.

# <font color='#E8800A'>3. Fill plans: compare, then select</font> <a class="anchor" id="plans"></a>
[Back to TOC](#toc)

**Setup, written once:**
- **The pipeline:** optional flag step → numeric `impute` → scaling, categorical `impute` → one-hot → model. The two `impute` steps are what the grid changes.
- **The instrument:** a logistic regression, as in the practical. It is a measuring instrument here, not a result.
- **KNN and iterative imputation** work on standardised columns. Distances and regressions depend on scale, so they get a scaler in front.

In [6]:
def add_first_purchase_flag(frame):
    """One 0/1 column: 1 for a first purchase, where every history NaN is structural."""
    return frame.assign(is_first_purchase=(frame["prior_purchase_count"] == 0).astype(int))


def build_pipeline(model):
    """The preprocessing under test. Every step can be swapped by a grid; `passthrough` steps do nothing.

    drop -> flag -> numeric: impute -> cap -> transform -> scale | categorical: impute -> encode -> model
    """
    numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("cap", "passthrough"),
                        ("transform", "passthrough"), ("scale", StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                            ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                                                     sparse_output=False))])
    prep = ColumnTransformer([("num", numeric, make_column_selector(dtype_include="number")),
                              ("cat", categorical, make_column_selector(dtype_exclude="number"))])
    return Pipeline([("drop", "passthrough"), ("flag", "passthrough"), ("prep", prep), ("model", model)])


def scaled(imputer):
    """KNN and iterative imputation measure distances / fit regressions: standardise first."""
    return make_pipeline(StandardScaler(), imputer)


logistic = LogisticRegression(max_iter=2000)
boosting = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

**The candidates.** Names read *numeric fill / categorical fill*:

| Fill | What it does |
|---|---|
| median, mean | training median / mean of the column |
| zero | 0, a value outside most columns' range, so it also marks the row |
| median + indicators | median, plus one 0/1 column per column that had NaNs (`add_indicator=True`) |
| KNN (k=5) | average of the 5 most similar training rows |
| iterative | predicts each column from the others (`IterativeImputer`) |
| mode / own level | most frequent label / a new label `"(missing)"` |

In [7]:
numeric_fills = {
    "median": SimpleImputer(strategy="median"),
    "mean": SimpleImputer(strategy="mean"),
    "zero": SimpleImputer(strategy="constant", fill_value=0),
    "median + indicators": SimpleImputer(strategy="median", add_indicator=True),
    "KNN (k=5)": scaled(KNNImputer(n_neighbors=5)),
    "iterative": scaled(IterativeImputer(random_state=RANDOM_STATE, max_iter=10)),
}
categorical_fills = {
    "mode": SimpleImputer(strategy="most_frequent"),
    "own level": SimpleImputer(strategy="constant", fill_value="(missing)"),
}
flags = {"no flag": "passthrough", "first-purchase flag": FunctionTransformer(add_first_purchase_flag)}
native_nan = {"native NaN": "passthrough"}  # no numeric fill: only for models that accept NaN

# Grid parameter -> the candidates it can take, to label the grid results by name
LABELS = {"prep__num__impute": {**numeric_fills, **native_nan},
          "prep__cat__impute": categorical_fills,
          "flag": flags}

In [8]:
def grid_scores(search, keys):
    """One row per fold, one column per plan (named after `keys`, in that order)."""
    results = search.cv_results_
    folds = sorted((c for c in results if re.fullmatch(r"split\d+_test_score", c)),
                   key=lambda c: int(re.search(r"\d+", c).group()))
    name = lambda key, obj: next(n for n, candidate in LABELS[key].items() if candidate is obj)
    labels = [" / ".join(name(key, params[key]) for key in keys) for params in results["params"]]
    return pd.DataFrame({label: [results[f][i] for f in folds] for i, label in enumerate(labels)},
                        index=pd.RangeIndex(1, len(folds) + 1, name="fold"))


def paired_table(scores, reference):
    """Each plan minus the reference ON THE SAME FOLD: mean difference, paired SEM, folds won."""
    diff = scores.sub(scores[reference], axis=0)
    table = pd.DataFrame({
        "ROC AUC": scores.mean(),
        "diff vs ref": diff.mean(),
        "paired SEM": diff.sem(),
        "diff / SEM": diff.mean() / diff.sem(),
        "folds won": (diff > 0).sum().astype(str) + f"/{len(scores)}",
    })
    table.loc[reference, ["diff vs ref", "paired SEM", "diff / SEM", "folds won"]] = [0.0, np.nan, np.nan, "ref"]
    return table.sort_values("ROC AUC", ascending=False)


def show(table):
    return table.style.format({"ROC AUC": "{:.4f}", "diff vs ref": "{:+.4f}", "paired SEM": "{:.4f}",
                               "diff / SEM": "{:+.1f}"}, na_rep="-")

**The grid:** 6 numeric fills × 2 categorical fills = 12 plans, each fitted on the 8 training folds. `GridSearchCV` runs the grid. `refit=False`, because we only want the per-fold scores, not a final model.

In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)  # IterativeImputer may stop before converging
    search_fill = GridSearchCV(
        build_pipeline(logistic),
        {"prep__num__impute": list(numeric_fills.values()),
         "prep__cat__impute": list(categorical_fills.values())},
        cv=prep_folds, scoring="roc_auc", refit=False, n_jobs=-1,
    ).fit(X, y)

fill_scores = grid_scores(search_fill, ["prep__num__impute", "prep__cat__impute"])
show(paired_table(fill_scores, "median / mode"))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
KNN (k=5) / own level,0.7841,+0.0023,0.0010,+2.2,7/8
zero / own level,0.7840,+0.0022,0.0009,+2.3,7/8
zero / mode,0.7835,+0.0016,0.0008,+1.9,7/8
KNN (k=5) / mode,0.7832,+0.0014,0.0010,+1.4,5/8
iterative / own level,0.7829,+0.0010,0.0013,+0.8,4/8
mean / own level,0.7828,+0.0010,0.0008,+1.2,6/8
median / own level,0.7828,+0.0009,0.0007,+1.2,6/8
iterative / mode,0.7821,+0.0002,0.0008,+0.3,3/8
median / mode,0.7819,+0.0000,-,-,ref
mean / mode,0.7815,-0.0003,0.0002,-1.5,1/8


**Reading the board** (each plan minus `median / mode`, fold by fold)
- **All 12 plans fall within 0.004 ROC AUC of each other.** How the gaps are filled barely matters for this model.
- **Best: `KNN / own level` (+0.0023 ± 0.0010) and `zero / own level` (+0.0022 ± 0.0009), about 2 SEM, ahead on 7 of 8 folds.** That is borderline, and it is the **best of 12 comparisons**: among 11 alternatives, one landing near 2 SEM is expected even when nothing really differs (the more candidates you test, the more one looks good by chance).
- **`own level` is ahead of `mode` for almost every numeric fill**, by about +0.001. Telling the model that a label was missing helps a little, but not clearly.
- **`zero`** probably gains for a semantic reason: for customers with only one prior purchase, a standard deviation of 0 is the true dispersion, not an arbitrary value.
- **KNN and iterative**, the elaborate fills, gain nothing clear over the median, as in the practical.
- **`median + indicators` is the only plan that loses** (−0.0017 ± 0.0008, 1 fold out of 8). The indicator columns duplicate information the model already has (see section 4).

**Decision: keep `median / mode`.** No plan wins clearly, and the median is the simplest plan to explain and reproduce. `zero / own level` stays as a candidate to re-check with the final models (section 6).

In [10]:
# The fill plan this section chose (numeric / categorical)
CHOSEN_NUMERIC = "median"
CHOSEN_CATEGORICAL = "mode"

# <font color='#E8800A'>4. First purchases: flag or not</font> <a class="anchor" id="structural"></a>
[Back to TOC](#toc)

**Question:** the 603 first purchases have NaN in every history column. Once filled with the median, they look like a "typical" returning customer. Does the model do better if it is **told** which rows are first purchases?

**Arms**
- **no flag:** the chosen fill only.
- **first-purchase flag:** the chosen fill + `is_first_purchase` (0/1).
- **zero instead of the median**, with and without the flag. Zero is outside the usual range of these columns, so it already marks the rows.
- **native NaN** (gradient boosting only): no numeric fill at all. `HistGradientBoostingClassifier` sends missing values to their own branch, which is a built-in flag.

**Two models:** the logistic regression, and gradient boosting, which reacts differently to fills and flags. A choice can help one family and not the other.

In [11]:
structural_grid = {"flag": list(flags.values()),
                   "prep__num__impute": [numeric_fills[CHOSEN_NUMERIC], numeric_fills["zero"]],
                   "prep__cat__impute": [categorical_fills[CHOSEN_CATEGORICAL]]}
keys = ["prep__num__impute", "flag"]
reference = f"{CHOSEN_NUMERIC} / no flag"

flag_scores = {}
for model_name, model, grid in [
    ("logistic regression", logistic, structural_grid),
    ("gradient boosting", boosting,
     {**structural_grid, "prep__num__impute": structural_grid["prep__num__impute"] + [native_nan["native NaN"]]}),
]:
    search = GridSearchCV(build_pipeline(model), grid, cv=prep_folds, scoring="roc_auc",
                          refit=False, n_jobs=-1).fit(X, y)
    flag_scores[model_name] = grid_scores(search, keys)

In [12]:
show(paired_table(flag_scores["logistic regression"], reference))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
zero / no flag,0.7835,+0.0016,0.0008,+1.9,7/8
zero / first-purchase flag,0.7821,+0.0002,0.0002,+0.9,4/8
median / first-purchase flag,0.7819,+0.0000,0.0005,+0.0,4/8
median / no flag,0.7819,+0.0000,-,-,ref


In [13]:
show(paired_table(flag_scores["gradient boosting"], reference))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
median / no flag,0.7730,+0.0000,-,-,ref
median / first-purchase flag,0.7730,+0.0000,0.0000,-,0/8
native NaN / no flag,0.7727,-0.0002,0.0028,-0.1,5/8
native NaN / first-purchase flag,0.7727,-0.0002,0.0028,-0.1,5/8
zero / no flag,0.7711,-0.0018,0.0035,-0.5,4/8
zero / first-purchase flag,0.7711,-0.0018,0.0035,-0.5,4/8


**Reading the results**
- **The first-purchase flag adds nothing, with either model**: +0.0000 for logistic regression, and an identical score for gradient boosting.
- The reason: the model **already knows** which rows are first purchases. `prior_purchase_count = 0` identifies them exactly, and it is a feature. The flag repeats information that is already there. The same explains why `median + indicators` did not help in section 3.
- **Gradient boosting with native NaN** scores like the median (−0.0002 ± 0.0028). Trees isolate the first purchases on their own with a split on `prior_purchase_count`.
- **`zero` helps logistic regression a little (+0.0016, 7/8 folds) and does not help gradient boosting** (−0.0018 ± 0.0035). A choice that works for one model family does not automatically work for the other.

**Decision: no flag**, and the same median fill for both families.

In [14]:
# The first-purchase treatment this section chose
CHOSEN_FLAG = "no flag"

# <font color='#E8800A'>5. Impute or drop</font> <a class="anchor" id="drop"></a>
[Back to TOC](#toc)

**Question:** instead of filling the scattered NaNs, drop the training rows that have them.

**Rules, as in the practical**
- Rows are dropped from the **training** side only. Every arm is scored on the **same validation rows**. Dropping hard validation rows would mean grading an easier exam.
- Only rows with a **scattered** NaN are dropped. Dropping rows with structural NaNs would delete every first purchase.

In [15]:
def chosen_pipeline(model):
    """The pipeline with the fill plan and first-purchase treatment chosen above."""
    return build_pipeline(model).set_params(
        flag=flags[CHOSEN_FLAG],
        prep__num__impute=clone(numeric_fills[CHOSEN_NUMERIC]),
        prep__cat__impute=clone(categorical_fills[CHOSEN_CATEGORICAL]),
    )


complete = ~rows_scattered.to_numpy()
drop_folds = [(tr[complete[tr]], va) for tr, va in prep_folds]  # fewer training rows, same validation rows

drop_scores = pd.DataFrame({
    "impute": evaluate(chosen_pipeline(logistic), prep_folds, X, y)[0]["ROC AUC"].to_numpy(),
    "drop rows with scattered NaN": evaluate(chosen_pipeline(logistic), drop_folds, X, y)[0]["ROC AUC"].to_numpy(),
}, index=pd.RangeIndex(1, len(prep_folds) + 1, name="fold"))

print(f"training rows dropped per fold: {np.mean([len(a) - len(b) for (a, _), (b, _) in zip(prep_folds, drop_folds)]):.0f} on average")
show(paired_table(drop_scores, "impute"))

training rows dropped per fold: 653 on average


,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
drop rows with scattered NaN,0.7830,+0.0011,0.0010,+1.1,4/8
impute,0.7819,+0.0000,-,-,ref


**Reading the results**
- Dropping the rows with scattered NaNs removes **~650 training rows per fold** and scores **+0.0011 ± 0.0010**: 1 SEM, ahead on only 4 of 8 folds. That is **no difference**.
- Imputing keeps those rows, and it is needed anyway: **test has scattered NaNs too** (30 rows per categorical), and test rows cannot be dropped.

**Decision: impute.**

# <font color='#E8800A'>6. Effect on the official validation</font> <a class="anchor" id="effect"></a>
[Back to TOC](#toc)

**Back to the official scheme** (3 yearly folds + 90-day gap) and to the Kaggle metric. Two pipelines, with both models:
- **chosen:** the plan selected above (it is also the V01 baseline);
- **runner-up:** `zero / own level`, the best plan of section 3. Does its small lead survive on the official folds?

- ROC AUC per fold.
- **Honest F1**: threshold chosen on folds 1–2, scored on fold 3, as for the V01 submission (CV 0.771, Kaggle 0.770).

In [16]:
def official_summary(pipeline):
    """ROC AUC mean over the official folds + honest F1 (threshold from folds 1-2, scored on fold 3)."""
    scores, oof = evaluate(pipeline, official_folds, X, y)
    early = np.concatenate([va for _, va in official_folds[:-1]])
    last = official_folds[-1][1]
    t_early, _ = best_threshold(y.iloc[early], oof.iloc[early])
    return {"ROC AUC mean": scores["ROC AUC"].mean(),
            "ROC AUC fold 1 / 2 / 3": " / ".join(f"{v:.3f}" for v in scores["ROC AUC"]),
            "threshold (folds 1-2)": t_early,
            "honest F1 (fold 3)": f1_score(y.iloc[last], oof.iloc[last] >= t_early)}


def runner_up_pipeline(model):
    return chosen_pipeline(model).set_params(prep__num__impute=clone(numeric_fills["zero"]),
                                             prep__cat__impute=clone(categorical_fills["own level"]))


effect = pd.DataFrame({
    (model_name, version): official_summary(pipeline(model))
    for model_name, model in [("logistic regression", logistic), ("gradient boosting", boosting)]
    for version, pipeline in [(f"chosen: {CHOSEN_NUMERIC} / {CHOSEN_CATEGORICAL}, {CHOSEN_FLAG}", chosen_pipeline),
                              ("runner-up: zero / own level", runner_up_pipeline)]
}).T
effect.style.format({"ROC AUC mean": "{:.4f}", "threshold (folds 1-2)": "{:.2f}", "honest F1 (fold 3)": "{:.3f}"})

**Reading the results**
- **The chosen plan is the V01 baseline**, so its numbers are the ones already known (logistic regression: ROC AUC 0.778, honest F1 0.771 → Kaggle 0.770).
- **The runner-up `zero / own level` does not confirm a clear advantage:**
  - logistic regression: +0.0016 ROC AUC (ahead on 2 folds, tied on 1), honest F1 0.776 vs 0.771;
  - gradient boosting: −0.0014 ROC AUC, ahead on 1 fold out of 3.
- An effect this small, whose sign depends on the model, is not a reason to switch. It is a **reasonable option to re-test as a hyperparameter** when tuning the linear models in `06_models`.

# <font color='#E8800A'>7. Summary of part 1</font> <a class="anchor" id="summary"></a>
[Back to TOC](#toc)

### Decisions on missing values

| Question | Decision | Evidence |
|---|---|---|
| How to fill numeric / categorical NaNs | **median / mode** | 12 plans within 0.004 ROC AUC. The best one (+0.0022, ~2 SEM) is the best of 12 and does not hold on the official folds for both models (§3, §6) |
| Flag first purchases | **no** | +0.0000 with both models: `prior_purchase_count = 0` already identifies them (§4) |
| Native NaN for gradient boosting | **not needed** | same score as the median (§4) |
| Impute or drop rows with scattered NaNs | **impute** | dropping 650 rows/fold: +0.0011 ± 0.0010, no difference. Test has NaNs too (§5) |
| To re-check later | `zero / own level` for linear models | +0.0016 ROC AUC for logistic regression, −0.0014 for boosting (§6) |

**Main message:** filling the missing values is **not where the score is**. Every reasonable plan gives about the same result. Whatever gain is left has to come from **the features** (`05_features`) and **the models** (`06_models`).

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Fill numeric NaNs with the training median, categorical NaNs with the training mode (inside the Pipeline) | mean, zero, median + indicators, KNN, iterative; own-level label | `preprocessing_ludovico` §3: 12 plans within 0.004 ROC AUC on 8 half-year time folds; no plan clearly better |
| No first-purchase flag | `is_first_purchase` column, per-column missing indicators | §4: +0.0000 for logistic regression and boosting; `prior_purchase_count = 0` already identifies first purchases |
| Impute, do not drop rows with missing values | drop training rows with scattered NaNs | §5: +0.0011 ± 0.0010 (no difference), and test rows cannot be dropped |

# Part 2 · Numeric columns

# <font color='#E8800A'>8. Skewness and scaling</font> <a class="anchor" id="skew"></a>
[Back to TOC](#toc)

**Question:** almost every numeric column is strongly right-skewed (EDA Step 13–14): many small values and a few very large ones. Models that use distances or a linear combination are dominated by those few large values. Which transformation and which scaler help?

**Who it concerns:** logistic regression and **KNN** are the measuring instruments here, one linear and one distance-based. Tree models are not tested. A monotonic transformation (log, quantiles) or a rescaling keeps the order of the values, so a tree finds the same splits.

**Candidates**, applied after imputation (*transformation / scaler*):

| Transformation | What it does |
|---|---|
| none | values as they are |
| log1p | `log(1 + x)`: compresses large values. Every numeric column is ≥ 0 after cleaning |
| Yeo-Johnson | `PowerTransformer`: chooses, per column and on the training fold, the power that makes it most symmetric |
| quantile → normal | `QuantileTransformer`: replaces each value with its rank, mapped to a normal distribution |

| Scaler | What it does |
|---|---|
| standard | subtract the mean, divide by the standard deviation |
| robust | subtract the median, divide by the interquartile range: less affected by extreme values |

In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, RobustScaler

knn = KNeighborsClassifier(n_neighbors=25)  # an instrument, not a tuned model

transforms = {
    "none": "passthrough",
    "log1p": FunctionTransformer(np.log1p),
    "Yeo-Johnson": PowerTransformer(method="yeo-johnson"),
    "quantile -> normal": QuantileTransformer(output_distribution="normal", n_quantiles=500,
                                              random_state=RANDOM_STATE),
}
scalers = {"standard": StandardScaler(), "robust": RobustScaler()}
LABELS.update({"prep__num__transform": transforms, "prep__num__scale": scalers})


def run_grid(model, grid, keys, base=None):
    """Per-fold ROC AUC of every plan of `grid` (on the 8 preprocessing folds)."""
    search = GridSearchCV(base if base is not None else build_pipeline(model), grid, cv=prep_folds,
                          scoring="roc_auc", refit=False, n_jobs=-1).fit(X, y)
    return grid_scores(search, keys)


skew_grid = {"prep__num__transform": list(transforms.values()), "prep__num__scale": list(scalers.values())}
skew_keys = ["prep__num__transform", "prep__num__scale"]
skew_scores = {name: run_grid(model, skew_grid, skew_keys)
               for name, model in [("logistic regression", logistic), ("KNN (k=25)", knn)]}
show(paired_table(skew_scores["logistic regression"], "none / standard"))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
log1p / robust,0.7870,+0.0051,0.0027,+1.9,5/8
Yeo-Johnson / robust,0.7869,+0.0050,0.0035,+1.4,5/8
log1p / standard,0.7866,+0.0047,0.0026,+1.8,5/8
Yeo-Johnson / standard,0.7865,+0.0046,0.0035,+1.3,5/8
none / robust,0.7820,+0.0001,0.0002,+0.5,5/8
none / standard,0.7819,+0.0000,-,-,ref
quantile -> normal / standard,0.7703,-0.0115,0.0068,-1.7,3/8
quantile -> normal / robust,0.7656,-0.0163,0.0070,-2.3,1/8


In [18]:
show(paired_table(skew_scores["KNN (k=25)"], "none / standard"))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
Yeo-Johnson / robust,0.7817,+0.0128,0.0057,+2.2,6/8
log1p / robust,0.7808,+0.0119,0.0059,+2.0,6/8
Yeo-Johnson / standard,0.7743,+0.0054,0.0057,+0.9,5/8
log1p / standard,0.7710,+0.0020,0.0062,+0.3,3/8
quantile -> normal / standard,0.7705,+0.0016,0.0078,+0.2,3/8
none / standard,0.7689,+0.0000,-,-,ref
none / robust,0.7689,-0.0001,0.0051,-0.0,5/8
quantile -> normal / robust,0.7664,-0.0025,0.0060,-0.4,3/8


**Reading the boards**
- **Compressing the large values helps both models.** `log1p` and Yeo-Johnson give about **+0.005** to logistic regression (1.3–1.9 SEM, 5/8 folds) and up to **+0.012** to KNN (about 2 SEM, 6/8 folds). Two different instruments move in the same direction, and the EDA predicted it (Step 14: `log1p` removes most of the skew). That is stronger evidence than a single ~2 SEM result.
- **The scaler matters for KNN, not for logistic regression.** With a transformation, `robust` is clearly better than `standard` for KNN (+0.012 vs +0.002–0.005). For logistic regression they are equivalent. Distances are sensitive to how each column is scaled, a linear model much less.
- **`quantile -> normal` makes things worse** (down to −0.016 for logistic regression). It turns every column into ranks, and the many zeros of these columns all collapse onto the same value.
- **`log1p` and Yeo-Johnson score the same.** `log1p` is simpler: no parameter to learn, and it is easy to explain.

**Decision: `log1p` + robust scaler** for linear and distance-based models.

In [19]:
# The numeric transformation this section chose (for linear and distance-based models)
CHOSEN_TRANSFORM = "log1p"
CHOSEN_SCALE = "robust"

# <font color='#E8800A'>9. Outliers</font> <a class="anchor" id="outliers"></a>
[Back to TOC](#toc)

**Question:** after the cleaning no value is *impossible*, but some are *extreme*: very large customers, very large invoices. Does capping them help?

**Candidate:** clip each numeric column to its 1st–99th (or 0.5th–99.5th) percentile, **computed on the training fold**. Validation values beyond the limits are clipped to the same limits.

It is tested with and without `log1p`. The log already compresses the extremes, so capping may become useless after it.

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin


class PercentileCapper(BaseEstimator, TransformerMixin):
    """Clip every column to percentiles learned on the training rows."""

    def __init__(self, lower=1, upper=99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.low_, self.high_ = np.nanpercentile(np.asarray(X, dtype=float), [self.lower, self.upper], axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low_, self.high_)


caps = {"no cap": "passthrough", "cap 1-99": PercentileCapper(1, 99), "cap 0.5-99.5": PercentileCapper(0.5, 99.5)}
LABELS["prep__num__cap"] = caps

cap_grid = {"prep__num__cap": list(caps.values()),
            "prep__num__transform": [transforms["none"], transforms["log1p"]],
            "prep__num__scale": [scalers[CHOSEN_SCALE]]}
cap_keys = ["prep__num__cap", "prep__num__transform"]
cap_scores = {name: run_grid(model, cap_grid, cap_keys)
              for name, model in [("logistic regression", logistic), ("KNN (k=25)", knn)]}
show(paired_table(cap_scores["logistic regression"], "no cap / none"))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
no cap / log1p,0.7870,+0.0050,0.0026,+1.9,5/8
cap 1-99 / log1p,0.7867,+0.0048,0.0026,+1.8,6/8
cap 0.5-99.5 / log1p,0.7865,+0.0046,0.0028,+1.6,5/8
no cap / none,0.7820,+0.0000,-,-,ref
cap 0.5-99.5 / none,0.7810,-0.0010,0.0015,-0.7,4/8
cap 1-99 / none,0.7802,-0.0017,0.0019,-0.9,2/8


In [21]:
show(paired_table(cap_scores["KNN (k=25)"], "no cap / none"))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
cap 1-99 / log1p,0.7824,+0.0135,0.0049,+2.7,6/8
cap 0.5-99.5 / log1p,0.7812,+0.0123,0.0051,+2.4,6/8
no cap / log1p,0.7808,+0.0120,0.0053,+2.2,6/8
cap 1-99 / none,0.7696,+0.0007,0.0025,+0.3,4/8
cap 0.5-99.5 / none,0.7696,+0.0007,0.0010,+0.7,5/8
no cap / none,0.7689,+0.0000,-,-,ref


**Reading the boards**
- **After `log1p`, capping changes nothing**: −0.0002 to −0.0005 for logistic regression, between −0.001 and +0.001 for KNN, all well under 1 SEM. The log already brings the extreme values close to the others.
- **Without the log, capping slightly hurts logistic regression** (about −0.0016, ahead on 2 of 8 folds). The extreme customers are real, and partly informative.

**Decision: no capping.** It adds a step and gains nothing once `log1p` is applied.

In [22]:
# The outlier treatment this section chose (for linear and distance-based models)
CHOSEN_CAP = "no cap"

# Part 3 · Categorical columns

# <font color='#E8800A'>10. Categorical columns</font> <a class="anchor" id="categorical"></a>
[Back to TOC](#toc)

**Questions** (EDA Step 15)
1. **Near-constant columns:** `warehouse` (99.7% one level) and `commercial_zone` (96.8%) carry almost no variation. Keep or drop?
2. **Rare levels:** `transport_method` has 80 levels, 51 of them with fewer than 30 rows. One-hot creates many columns that are almost always 0.

**Candidates for the encoding** (applied to all 6 categorical columns):

| Encoding | What it does |
|---|---|
| one-hot (min 1) | one column per level, no grouping |
| one-hot (min 20) / (min 50) | levels with fewer than 20 / 50 training rows grouped into one "infrequent" column (current choice: min 20) |
| target encoding | each level replaced by the **share of positives** of that level in the training fold. `TargetEncoder` computes it with an internal cross-fitting, so a row never sees its own target |

**Unseen levels in test** (`commercial_zone` alone has 209 test rows with a level never seen in train): one-hot maps them to the "infrequent" column, target encoding to the overall rate. Both handle them without errors.

**Models:** logistic regression and gradient boosting. Encoding matters for both, in different ways.

In [23]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder

NEAR_CONSTANT = ["warehouse", "commercial_zone"]


def drop_near_constant(frame):
    return frame.drop(columns=NEAR_CONSTANT)


encoders = {
    "one-hot (min 1)": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "one-hot (min 20)": OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False),
    "one-hot (min 50)": OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=False),
    "target encoding": TargetEncoder(target_type="binary", cv=KFold(5, shuffle=True, random_state=RANDOM_STATE)),
}
drops = {"keep all": "passthrough", "drop near-constant": FunctionTransformer(drop_near_constant)}
LABELS.update({"prep__cat__encode": encoders, "drop": drops})

cat_grid = {"prep__cat__encode": list(encoders.values()), "drop": list(drops.values())}
cat_keys = ["prep__cat__encode", "drop"]
cat_reference = "one-hot (min 20) / keep all"

linear_base = build_pipeline(logistic).set_params(prep__num__transform=transforms[CHOSEN_TRANSFORM],
                                                  prep__num__scale=scalers[CHOSEN_SCALE],
                                                  prep__num__cap=caps[CHOSEN_CAP])
cat_scores = {
    "logistic regression": run_grid(logistic, cat_grid, cat_keys, base=linear_base),
    "gradient boosting": run_grid(boosting, cat_grid, cat_keys),
}
show(paired_table(cat_scores["logistic regression"], cat_reference))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
one-hot (min 50) / drop near-constant,0.7916,+0.0047,0.0021,+2.3,7/8
one-hot (min 50) / keep all,0.7910,+0.0040,0.0020,+2.0,6/8
one-hot (min 1) / drop near-constant,0.7897,+0.0027,0.0022,+1.2,4/8
target encoding / keep all,0.7893,+0.0024,0.0036,+0.7,5/8
target encoding / drop near-constant,0.7893,+0.0023,0.0035,+0.6,5/8
one-hot (min 1) / keep all,0.7885,+0.0016,0.0022,+0.7,4/8
one-hot (min 20) / drop near-constant,0.7881,+0.0011,0.0005,+2.1,5/8
one-hot (min 20) / keep all,0.7870,+0.0000,-,-,ref


In [24]:
show(paired_table(cat_scores["gradient boosting"], cat_reference))

,ROC AUC,diff vs ref,paired SEM,diff / SEM,folds won
one-hot (min 50) / keep all,0.7756,+0.0026,0.0027,+1.0,4/8
target encoding / drop near-constant,0.7753,+0.0023,0.0025,+1.0,5/8
one-hot (min 20) / drop near-constant,0.7749,+0.0019,0.0020,+1.0,6/8
one-hot (min 1) / keep all,0.7736,+0.0006,0.0022,+0.3,4/8
target encoding / keep all,0.7734,+0.0004,0.0045,+0.1,4/8
one-hot (min 20) / keep all,0.7730,+0.0000,-,-,ref
one-hot (min 50) / drop near-constant,0.7722,-0.0007,0.0019,-0.4,3/8
one-hot (min 1) / drop near-constant,0.7695,-0.0035,0.0034,-1.0,3/8


**Reading the boards** (logistic regression with `log1p` + robust)
- **Grouping the rare levels more strongly helps logistic regression a little:** `one-hot (min 50)` is at **+0.0040 ± 0.0020** (2.0 SEM, 6/8 folds). It also goes in the same direction for gradient boosting (+0.0026, 1 SEM). Fewer and better-populated columns give more stable coefficients.
- **Target encoding gains nothing clear** for either model (+0.0024 ± 0.0036 for logistic regression, +0.0004 ± 0.0045 for boosting). Its scores also move by about ±0.002 when only the shuffling of its internal cross-fitting changes. It is less stable, harder to explain, and needs care against leakage.
- **Dropping the near-constant columns is not consistent**: about +0.001 for logistic regression, and both positive and negative for boosting depending on the encoding. These columns are almost constant, so they barely affect the model either way. Whether to drop them belongs to the **feature selection** (`05_features`).

**Decision: one-hot with levels under 50 training rows grouped as "infrequent", for both families. Keep all columns** (to be revisited in feature selection).

In [25]:
# The categorical treatment this section chose
CHOSEN_ENCODE = "one-hot (min 50)"
CHOSEN_DROP = "keep all"

# <font color='#E8800A'>11. The two recipes on the official validation</font> <a class="anchor" id="recipes"></a>
[Back to TOC](#toc)

**The result of the notebook: two preprocessing recipes**, one per model family. They are compared with the V01 baseline on the **official scheme** (3 yearly folds + 90-day gap) and the Kaggle metric (honest F1).
- **Linear / distance recipe** (logistic regression, KNN, and later SVM / neural networks): every choice of sections 3–10.
- **Tree recipe** (decision tree, random forest, boosting): the same fill and categorical choices, without transformation, capping or scaling, which do not change a tree.

In [26]:
def linear_recipe(model):
    """Preprocessing for linear and distance-based models."""
    return chosen_pipeline(model).set_params(
        prep__num__cap=clone(caps[CHOSEN_CAP]) if CHOSEN_CAP != "no cap" else "passthrough",
        prep__num__transform=clone(transforms[CHOSEN_TRANSFORM]) if CHOSEN_TRANSFORM != "none" else "passthrough",
        prep__num__scale=clone(scalers[CHOSEN_SCALE]),
        prep__cat__encode=clone(encoders[CHOSEN_ENCODE]),
        drop=drops[CHOSEN_DROP],
    )


def tree_recipe(model):
    """Preprocessing for tree models: same fills and categorical choices, no numeric reshaping."""
    return chosen_pipeline(model).set_params(
        prep__num__scale="passthrough",
        prep__cat__encode=clone(encoders[CHOSEN_ENCODE]),
        drop=drops[CHOSEN_DROP],
    )


recipes = pd.DataFrame({
    (model_name, version): official_summary(pipeline(model))
    for model_name, model, recipe in [("logistic regression", logistic, linear_recipe),
                                      ("KNN (k=25)", knn, linear_recipe),
                                      ("gradient boosting", boosting, tree_recipe)]
    for version, pipeline in [("baseline (V01 preprocessing)", build_pipeline), ("recipe", recipe)]
}).T
recipes.style.format({"ROC AUC mean": "{:.4f}", "threshold (folds 1-2)": "{:.2f}", "honest F1 (fold 3)": "{:.3f}"})

**Reading the results** (official scheme: 3 yearly folds, 90-day gap)

| Model | ROC AUC baseline → recipe | Folds improved | Honest F1 baseline → recipe |
|---|---|---|---|
| logistic regression | 0.778 → **0.783** (+0.005) | 3/3 | 0.771 → **0.781** |
| KNN (k=25) | 0.761 → **0.776** (+0.015) | 3/3 | 0.770 → **0.787** |
| gradient boosting | 0.759 → 0.763 (+0.004) | 2/3 (1 tied) | 0.771 → 0.764 |

- **The linear recipe works on the official folds as well**, every fold up for both models. The choices made on the 8 half-year folds hold on a different split of the same period.
- **KNN gains the most**: distance-based models are the ones that depend most on preprocessing.
- **Gradient boosting barely moves**, as expected: only the categorical grouping changes for it. Differences of a few thousandths come from the coarser one-hot and from numerical rounding.
- **The honest F1 is a single fold with a threshold chosen on the other two, so it is noisy** (the threshold for boosting jumps to 0.18). Use ROC AUC to compare preprocessing, and F1 only as a check.

# <font color='#E8800A'>12. Summary</font> <a class="anchor" id="final"></a>
[Back to TOC](#toc)

### The two preprocessing recipes

| Step | Linear / distance recipe (logistic, KNN, SVM, NN) | Tree recipe (tree, random forest, boosting) | Evidence |
|---|---|---|---|
| Numeric NaNs | training median | training median | §3: 12 fills within 0.004 ROC AUC |
| Categorical NaNs | training mode | training mode | §3 |
| First-purchase flag | no | no | §4: +0.0000, `prior_purchase_count` already says it |
| Transformation | **`log1p`** | none | §8: +0.005 logistic, +0.012 KNN |
| Outlier capping | no | no | §9: no gain after `log1p` |
| Scaling | **robust** | none | §8: robust ≫ standard for KNN |
| Categorical encoding | **one-hot, levels < 50 rows grouped** | same | §10: +0.004 logistic, same direction for boosting |
| Near-constant columns | kept (decided in feature selection) | kept | §10: no consistent effect |

**On the official validation:** logistic regression 0.778 → 0.783 ROC AUC and KNN 0.761 → 0.776, every fold up. Gradient boosting is essentially unchanged.

**Main message:** filling the NaNs does not matter. **Reshaping the numeric columns does**, for linear and distance-based models. Trees need almost nothing.

### Lines for `docs/decisions.md`

| Decision | Alternatives considered | Evidence |
|---|---|---|
| Linear/distance models: `log1p` on numeric columns + `RobustScaler` | none, Yeo-Johnson, quantile→normal; `StandardScaler` | `preprocessing_ludovico` §8: +0.005 ROC AUC logistic, +0.012 KNN (≈2 SEM, 6/8 folds); official folds 3/3 up (§11) |
| No outlier capping | capping at 1–99 / 0.5–99.5 training percentiles | §9: no gain after `log1p`, slight loss without it |
| One-hot encoding, levels with < 50 training rows grouped as "infrequent" | min 1 / min 20, target encoding | §10: +0.004 ± 0.002 logistic, same direction boosting; target encoding no clear gain |
| Tree models: no transformation or scaling | same recipe as linear | monotonic transforms do not change tree splits; §11 boosting unchanged |

### Next step
**`05_features`**: new features (recency relative to the usual gap, activity trend between windows, margin, discount share) and **feature selection** (including the near-constant columns and the redundant monetary ones). The two recipes are the starting point.